# 第18讲：告警诊断的公平业务评测

先校准评分器，再比较原版、多告警候选版与漏集群过滤的反例版。数据均为构造教学数据，题面和停止方式见 [README](README.md)。真实执行使用 macOS 沙箱；评分器在沙箱之外。


In [ ]:
from pathlib import Path
import json
import sys

HERE = Path.cwd()
if not (HERE / 'alert_workshop.py').is_file():
    HERE = HERE / 'examples/18-business-eval'
assert (HERE / 'alert_workshop.py').is_file(), '请从本讲或仓库根目录打开'
sys.path.insert(0, str(HERE))
from alert_dataset import CASES, public_input
from alert_scorer import grade
from alert_selftest import correct, run
from alert_workshop import AlertEvaluation, compare

[(item['id'], item['group'], item['task']) for item in CASES]


## 1. 先证明评分器能区分什么

此格不调用模型。已知正确与错误的轨迹只用于评分校准，绝不充当模型回答。检查跨集群、错根因、虚构时间、漏查询、请求与回执错配，以及基础设施故障的空评分。


In [ ]:
controls = run()
assert controls['passed']
known_correct = correct(CASES[0])
print(grade(CASES[0], known_correct)['score'])
known_failure = {**known_correct, 'infrastructure_error': '构造连接中断'}
assert grade(CASES[0], known_failure)['score'] is None
controls


## 2. 原生调用在哪里执行

不能在持有参考答案的当前内核里运行被测 Agent。`execute()` 经 `sandbox-exec` 启动 [alert_worker.py](alert_worker.py)，它使用 [hermes_worker.py](hermes_worker.py) 中的真实 Hermes 调用。下面是关键原生调用，不是另一套模拟 Agent：

```python
db = SessionDB(home / 'state.db')
agent = AIAgent(model=os.environ['EVAL_MODEL'], provider='custom', api_mode='chat_completions',
    api_key=os.environ['GLM_API_KEY'], base_url=os.environ['EVAL_API'],
    enabled_toolsets=['skills', 'terminal'],
    max_iterations=int(os.environ.get('EVAL_MAX_ITERATIONS', 12)),
    max_tokens=int(os.environ.get('EVAL_MAX_TOKENS', 4096)),
    request_overrides={'temperature': float(os.environ['EVAL_TEMPERATURE']),
                       'extra_body': {'thinking': {'type': 'disabled'}}},
    quiet_mode=True, skip_context_files=True, load_soul_identity=False,
    save_trajectories=False, session_id=sid, session_db=db,
    tool_start_callback=start, tool_complete_callback=end)
try:
    result = agent.run_conversation(user_message=prompt['task'], conversation_history=[],
                                   system_message=prompt['system'], task_id=sid)
finally:
    agent.close()
    db.close()
```

模型只能读取本版技能并调用 `./incidentctl alerts|logs|traces CLUSTER SERVICE`。课程适配器通过原生 `registry.register` 接入只读查询；工具白名单与操作系统沙箱共同阻止执行者读取题库答案和修改技能。每题每版都有新 Home、SessionDB、同一预置记忆、同一输入及快照。


In [ ]:
evaluation = AlertEvaluation()
print(evaluation.protocol['connection'])
print('本轮目录：', evaluation.root)


## 3. 开始真实对照

下一格需要 GLM 凭据并产生费用：4题 × 3版 × 2次，共24个独立会话，顺序执行。原版与候选的不同只能来自技能。反例版可能被模型纠正，不预设它必然失败。中断内核会终止当前执行进程组；重新执行请从新内核、新运行开始，不覆盖旧行。


In [ ]:
for repeat in range(1, evaluation.protocol['repeats'] + 1):
    for item in CASES:
        for version in evaluation.protocol['versions']:
            row = evaluation.execute(version, item, repeat)
            print(version, item['id'], repeat, row['grading']['status'], row['grading']['score'])


## 4. 查看回执，不只看分数

评分在当前父进程中执行。检查一次真实终端请求与同编号回执，确认集群过滤和事故归属；不要把说明中提到某个记录当成已经查询过。


In [ ]:
from alert_scorer import pairs
first = evaluation.rows[0]
item = next(item for item in CASES if item['id'] == first['case_id'])
print(grade(item, first['execution']))
pairs(first['execution'].get('messages', []))


## 5. 逐题比较与采用判断

输入、快照、评分器及协议指纹不一致或评分为空时，结果不可比较。程序不会因均分改善而掩盖关键旧题退步，也不会自动采用候选。是否值得采用还要人工阅读因果说明与成本。如果修改评分器，应在同一规则下重新评测两版，不改旧回放哈希冒充匹配。


In [ ]:
comparison = compare(evaluation.rows, 'after', evaluation.protocol['repeats'])
print(comparison)
result = evaluation.finish()
result
